# EDGAR Credit Clustering v3 — Clean src-driven notebook

This notebook is intentionally thin. It orchestrates the workflow and delegates reusable logic to `src.credit_clustering`:

- EDGAR concept/SIC mapping: `edgar_concepts.py`
- Feature engineering: `features.py`
- Clustering: `clustering.py`
- Cluster profiling: `profiling.py`
- Artifact building/saving: `artifacts.py`

The notebook should not define reusable methods.

In [1]:
import os
import sys
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)

In [2]:
BASE_URL = "https://pub-a6c3a3e1a0f546beb4be7cc34fd647d1.r2.dev/raw_financial_facts_parquet"

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DUCKDB_DIR = PROJECT_ROOT / "data" / "duckdb"
DUCKDB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DUCKDB_DIR / "financials.duckdb"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_DIR = OUTPUT_DIR / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)

CURRENT_OUTPUT_DIR = OUTPUT_DIR / "credit_clustering_outputs_v3"
CURRENT_OUTPUT_DIR.mkdir(exist_ok=True)

ARTIFACT_PATH = MODEL_DIR / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"

In [3]:
#Import your modules here

from src.credit_clustering import (
    # Config
    PUBLIC_COMPANY_MIN_ASSETS,
    SCORECARD_CLUSTER_FEATURES,
    SCORECARD_COMPONENT_FEATURES,
    RATIO_COLS,
    INTERPRET_FEATURES,
    DEFAULT_SEGMENT_COL,
    DEFAULT_TARGET_SEGMENTS,
    DEFAULT_N_CLUSTERS,
    DEFAULT_MIN_ROWS_PER_SEGMENT,
    cluster_segments,

    # EDGAR concepts and panel construction
    EDGAR_CONCEPT_MAP,
    create_issuer_year_facts_table,
    build_issuer_year_panel,

    # Feature engineering
    engineer_private_company_features,

    # Clustering
    cluster_segments,
    evaluate_segments_k_range,

    # Profiling
    build_cluster_profile,
    build_cluster_medians,
    build_feature_extremes,
    build_industry_cluster_mix,
    add_rating_style_labels,
    build_rating_label_maps,
    representatives,
    merge_profile_with_representatives,

    # Artifacts
    build_credit_clustering_artifact,
    save_artifact,
    summarize_artifact,

    #Config settings
    DEFAULT_N_CLUSTERS,
    DEFAULT_RANDOM_STATE,
    DEFAULT_N_INIT,
)

## 1. Connect to EDGAR raw facts

The raw EDGAR-derived facts are read from remote Parquet files into a DuckDB view.

In [4]:
parquet_urls = [
    f"{BASE_URL}/part_{i:05d}.parquet"
    for i in range(401)
]

con = duckdb.connect(DB_PATH)

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute(f"""
CREATE OR REPLACE VIEW raw_facts AS
SELECT *
FROM read_parquet({parquet_urls})
""")

schema = con.execute("DESCRIBE raw_facts").df()
schema.head()

,column_name,column_type,null,key,default,extra
0,concept,VARCHAR,YES,None,None,None
1,label,VARCHAR,YES,None,None,None
2,value,DOUBLE,YES,None,None,None
3,numeric_value,DOUBLE,YES,None,None,None
4,unit,VARCHAR,YES,None,None,None


In [5]:
#Optional: Inspect the whole existing database. Attention this is will call the database and
#take close to or over 1 minute. It is not needed downstream. 
RUN_RAW_DATABASE_AUDIT = False

if RUN_RAW_DATABASE_AUDIT:
    raw_summary = con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT ticker) AS tickers,
        COUNT(DISTINCT concept) AS concepts,
        MIN(fiscal_year) AS min_year,
        MAX(fiscal_year) AS max_year
    FROM raw_facts
    """).df()

    display(raw_summary)

## 2. Build issuer-year facts and panel

The EDGAR concept map lives in `src.credit_clustering.edgar_concepts`. The notebook only calls the extraction helpers.

In [6]:
facts_summary, value_col, sort_col = create_issuer_year_facts_table(
    con=con,
    schema=schema,
    concept_map=EDGAR_CONCEPT_MAP,
    start_year=2020,
    end_year=2025,
    fiscal_period="FY",
)

print("value_col:", value_col, "| sort_col:", sort_col)
display(facts_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

value_col: numeric_value | sort_col: period_end


,canonical_feature,row_count,ticker_count
0,net_income,36055,7171
1,assets,36044,7147
2,cfo,35527,7106
3,equity,35594,7048
4,cash,34942,7035
5,depreciation_amortization,31723,6491
6,liabilities,31638,6462
7,revenue,27296,5841
8,operating_income,27902,5778
9,ppe,26720,5771


In [7]:
panel = build_issuer_year_panel(
    con=con,
    concept_map=EDGAR_CONCEPT_MAP,
)

print("Panel shape:", panel.shape)
display(panel.head())

Panel shape: (36290, 29)


,ticker,cik,company_name,sic,fiscal_year,assets,assets_current,capex,cash,cfo,depreciation_amortization,equity,goodwill,gross_profit,interest_expense,inventory,liabilities,liabilities_current,long_term_debt,net_income,operating_income,ppe,rd,receivables,revenue,sga,short_term_debt,major_sector,financial_flag
0,THRY,1556739,"Thryv Holdings, Inc.",7310,2021,1.257740e+09,3.701715e+08,NaN,6.834000e+06,2.327720e+08,1.054730e+08,2.557450e+08,6.094570e+08,7.053390e+08,68539000.0,NaN,NaN,2.572670e+08,5.284030e+08,1.015770e+08,190013000.0,6.999100e+07,NaN,287811500.0,1.113382e+09,NaN,35000000.0,Services,Non-financial
1,ADSK,769397,"Autodesk, Inc.",7372,2023,9.022500e+09,3.052500e+09,56000000.0,1.773500e+09,1.531000e+09,5.100000e+07,9.070000e+08,3.614500e+09,3.968000e+09,NaN,NaN,NaN,4.004500e+09,2.300000e+09,8.230000e+08,629000000.0,1.530000e+08,1.115000e+09,838500000.0,4.386000e+09,NaN,NaN,Services,Non-financial
2,IMAX,921582,IMAX CORP,3861,2021,9.404985e+08,NaN,3590000.0,2.535450e+08,6.065000e+06,4.955000e+06,4.296140e+08,3.902700e+07,1.344060e+08,7010000.0,33252000.0,4.929290e+08,NaN,1.540740e+08,-1.595300e+07,10984000.0,2.688750e+08,5.618000e+06,83175000.0,2.548830e+08,NaN,NaN,Manufacturing / Industrials,Non-financial
3,IMO,49938,IMPERIAL OIL LTD,2911,2022,4.078200e+10,9.274500e+09,NaN,2.153000e+09,5.476000e+09,1.977000e+09,2.173500e+10,1.660000e+08,NaN,60000000.0,NaN,2.007900e+10,7.226000e+09,3.947000e+09,2.479000e+09,NaN,3.124000e+10,8.900000e+07,NaN,3.759000e+10,NaN,NaN,Manufacturing / Industrials,Non-financial
4,PSPX,1765651,Pacific Sports Exchange Inc.,5940,2021,1.854500e+04,1.854500e+04,NaN,2.208500e+04,-1.853900e+04,NaN,-9.850000e+02,NaN,3.049500e+03,NaN,1946.5,NaN,4.386300e+04,NaN,-4.736950e+04,-47369.5,NaN,NaN,NaN,NaN,NaN,NaN,Wholesale / Retail,Non-financial


In [8]:
panel.groupby(["major_sector", "fiscal_year"]).size().unstack(fill_value=0)

fiscal_year,2020,2021,2022,2023,2024,2025
major_sector,,,,,,
Agriculture,23,33,38,42,42,34
Construction,53,62,66,70,76,80
Finance / Insurance / Real Estate,1441,1522,1582,1621,1654,1663
Manufacturing / Industrials,1830,2099,2237,2297,2365,2253
Mining / Energy,213,225,248,254,246,239
Services,875,1060,1120,1156,1228,1187
Transportation / Utilities,425,450,470,470,487,479
Wholesale / Retail,326,378,388,400,410,373


## 3. Engineer credit scorecard features

Feature construction is centralized in `features.py` so training and private-company scoring use the same logic.

In [9]:
model_df = engineer_private_company_features(
    panel,
    winsor_caps=None,
    fx_to_model_currency=1.0,
)

# Training/reference universe filter.
# The threshold is controlled in config.py.
model_df = model_df.loc[
    model_df["assets"].ge(PUBLIC_COMPANY_MIN_ASSETS)
].copy()

print("Model rows:", len(model_df))
display(model_df[SCORECARD_CLUSTER_FEATURES].describe().T)

Model rows: 34299


,count,mean,std,min,25%,50%,75%,max
structural_distress_risk,34299.0,0.128663,0.334831,0.0,0.000000,0.000000,0.000000,1.0
earnings_risk,34117.0,0.526047,0.388227,0.0,0.175940,0.458806,1.000000,1.0
operating_cashflow_risk,33670.0,0.516182,0.403568,0.0,0.038914,0.568563,1.000000,1.0
liquidity_risk,25178.0,0.341250,0.353563,0.0,0.000000,0.208893,0.658163,1.0
leverage_risk,16415.0,0.323732,0.283252,0.0,0.058439,0.276197,0.527413,1.0
debt_service_risk,7791.0,0.504068,0.397345,0.0,0.063221,0.528076,0.955778,1.0


In [10]:
feature_availability = (
    model_df[SCORECARD_CLUSTER_FEATURES + SCORECARD_COMPONENT_FEATURES]
    .notna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("availability_pct")
)

feature_availability

,availability_pct
structural_distress_risk,1.000000
negative_equity_flag,1.000000
liabilities_exceed_assets_flag,1.000000
earnings_risk,0.994694
profitability_risk,0.994694
equity_buffer_risk,0.983323
operating_cashflow_risk,0.981661
cashflow_risk,0.981661
cash_buffer_risk,0.970407
liabilities_risk,0.878101


## 4. Cluster non-financial issuers

The clustering code lives in `clustering.py`. The model uses bounded directional risk factors, not raw accounting values.

In [11]:
model = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("cluster", KMeans(
                    n_clusters=DEFAULT_N_CLUSTERS,
                    init="k-means++",
                    n_init=DEFAULT_N_INIT,
                    random_state=DEFAULT_RANDOM_STATE,
                    verbose=True),
            ),
        ])

model

,steps,"[('imputer', ...), ('cluster', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,n_clusters,5


In [12]:
clustered_panel, metrics_df, segment_artifacts = cluster_segments(
    model_df,
    model=model,
    segment_col=DEFAULT_SEGMENT_COL,
    min_rows=DEFAULT_MIN_ROWS_PER_SEGMENT,
    cluster_only_segments=DEFAULT_TARGET_SEGMENTS,
)

print("Clustered rows:", len(clustered_panel))
display(metrics_df)

Initialization complete
Iteration 0, inertia 8996.806889376758.
Iteration 1, inertia 5521.343236074058.
Iteration 2, inertia 5336.787929475742.
Iteration 3, inertia 5315.941582361676.
Iteration 4, inertia 5311.9656192087905.
Iteration 5, inertia 5310.5274510884865.
Iteration 6, inertia 5309.926523464028.
Iteration 7, inertia 5309.494355030517.
Iteration 8, inertia 5309.216254470213.
Iteration 9, inertia 5308.960208197316.
Iteration 10, inertia 5308.758458959228.
Iteration 11, inertia 5308.648131238173.
Converged at iteration 11: center shift 8.73793212708817e-06 within tolerance 1.1780856404446312e-05.
Initialization complete
Iteration 0, inertia 8005.418344783331.
Iteration 1, inertia 5762.75648005476.
Iteration 2, inertia 5596.081927967307.
Iteration 3, inertia 5562.002342315732.
Iteration 4, inertia 5542.985659564716.
Iteration 5, inertia 5442.440826829983.
Iteration 6, inertia 5328.002063184706.
Iteration 7, inertia 5316.918280623968.
Iteration 8, inertia 5316.149166800842.
Iterati

,segment,status,rows,features,feature_list,silhouette,calinski_harabasz,davies_bouldin,feature_availability,min_non_null_features
0,Financial,skipped_not_target_segment,9271,0,[],NaN,NaN,NaN,NaN,NaN
1,Non-financial,clustered,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",0.360739,13416.41083,1.211267,"{'structural_distress_risk': 1.0, 'earnings_risk': 0.9956448777369347, 'operating_cashflow_risk': 0.9817004954451015, 'liquidity_risk': 0.9513744606041233, ...",4.0


In [13]:
cluster_sizes = (
    clustered_panel
    .groupby([DEFAULT_SEGMENT_COL, "cluster"])
    .size()
    .reset_index(name="issuer_years")
    .sort_values([DEFAULT_SEGMENT_COL, "cluster"])
)

cluster_sizes

,financial_flag,cluster,issuer_years
0,Non-financial,0,3350
1,Non-financial,1,7086
2,Non-financial,2,6593
3,Non-financial,3,2630
4,Non-financial,4,4344


## 5. Build cluster profiles and rating-style labels

The interpretation layer lives in `profiling.py`.

In [14]:
cluster_profile = build_cluster_profile(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_medians = build_cluster_medians(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

feature_extremes = build_feature_extremes(
    clustered_panel,
)

industry_cluster_mix = build_industry_cluster_mix(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_profile_ranked = add_rating_style_labels(
    cluster_profile,
    segment_col=DEFAULT_SEGMENT_COL,
)

rating_labels_by_cluster, risk_rank_by_cluster = build_rating_label_maps(
    cluster_profile_ranked,
    segment_name="Non-financial",
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_representatives = representatives(
    clustered_panel,
    segment_col=DEFAULT_SEGMENT_COL,
)

cluster_profile_ranked_with_reps = merge_profile_with_representatives(
    cluster_profile_ranked,
    cluster_representatives,
    segment_col=DEFAULT_SEGMENT_COL,
)

display(cluster_profile_ranked_with_reps)

,financial_flag,cluster,issuer_years,issuers,median_log_assets,median_liabilities_to_assets,median_debt_to_assets,median_debt_to_equity,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_revenue_to_assets,median_current_ratio,median_quick_ratio,median_interest_coverage,median_fcf_to_debt,median_operating_margin,median_gross_margin,median_ebitda_margin,median_debt_to_ebitda,median_net_debt_to_ebitda,median_ebitda_interest_coverage,median_leverage_risk,median_liquidity_risk,median_earnings_risk,median_operating_cashflow_risk,median_debt_service_risk,median_structural_distress_risk,median_scorecard_credit_score,rating_style_rank,rating_style_label,representative_tickers,sample_companies,sample_years
0,Non-financial,1,7086,1948,21.192826,0.453712,0.235620,0.482842,0.485157,0.117427,0.048957,0.093804,0.610141,2.367342,1.275503,7.003767,0.280404,0.107978,0.376422,0.148089,2.374761,1.248718,10.204706,0.071832,0.067100,0.010426,0.000000,0.087693,0.0,11.201168,1,1 - Strong relative credit profile,"CRTO, TREX, TECH, PKG, UG, NEOG, DXCM, KTEL, INOD, MZTI",Criteo S.A. | TREX CO INC | BIO-TECHNE Corp | PACKAGING CORP OF AMERICA | UNITED GUARDIAN INC,"2021, 2022, 2025, 2023, 2024"
1,Non-financial,4,4344,1241,22.807142,0.670743,0.294821,0.858678,0.301391,0.030583,0.033468,0.081292,0.491363,1.010816,0.350702,3.279628,0.202889,0.128044,0.329195,0.185624,3.372739,2.863957,5.418629,0.266252,0.737132,0.165317,0.000000,0.209970,0.0,32.467486,2,2 - Good credit profile,"NOA, YOU, LHX, SGU, SDSYA, MTN, CSTM, GLP-PB, GLP, OXYWS","North American Construction Group Ltd. | Clear Secure, Inc. | L3HARRIS TECHNOLOGIES, INC. /DE/ | STAR GROUP, L.P. | SOUTH DAKOTA SOYBEAN PROCESSORS LLC","2024, 2022, 2025, 2020"
2,Non-financial,2,6593,2032,18.683641,0.299129,0.126724,0.259562,0.600237,0.277450,-0.258140,-0.198351,0.240246,4.310476,2.130207,-27.662528,-1.220166,-0.818926,0.406889,-0.697438,9.780351,3.997271,-23.475050,0.039020,0.000000,1.000000,1.000000,1.000000,0.0,58.333333,3,3 - Leveraged / elevated risk profile,"WLDS, ROMA, NMTC, ATPC, KROS, CVKD, DNABW, VLN-WT, AUPH, GLTO","Wearable Devices Ltd. | Roma Green Finance Ltd | NEUROONE MEDICAL TECHNOLOGIES Corp | Agape ATP Corp | Keros Therapeutics, Inc.","2025, 2022, 2023, 2020"
3,Non-financial,3,2630,1269,19.568362,0.651929,0.243719,0.835501,0.296061,0.048820,-0.068510,-0.008554,0.457442,1.065136,0.401937,-2.720981,-0.092972,-0.118166,0.295310,-0.062674,9.130894,7.806352,-1.447577,0.289392,0.667038,1.000000,0.805037,0.905076,0.0,62.549707,4,4 - Weak credit profile,"TCX, VNCE, SMWB, CCC, SCOR, GITS, FARM, AIOS, NISN, NOTE","TUCOWS INC /PA/ | VINCE HOLDING CORP. | SIMILARWEB LTD. | CCC Intelligent Solutions Holdings Inc. | COMSCORE, INC.","2024, 2020, 2021, 2023, 2022"
4,Non-financial,0,3350,1572,18.060449,1.086288,0.396340,-0.805166,-0.209223,0.103684,-0.277459,-0.151918,0.451918,0.846273,0.393565,-7.905898,-0.521353,-0.529271,0.331066,-0.459393,9.587202,6.870450,-6.715322,0.693849,0.709803,1.000000,1.000000,1.000000,1.0,77.597603,5,5 - Distressed / near-default proxy,"OCLN, NUAI, NUAIW, LVO, UXIN, CRTD, FOXOW, FOXO, MDAI, MDAIW","ORIGINCLEAR, INC. | New ERA Energy & Digital, Inc. | LiveOne, Inc. | Uxin Ltd | Creatd, Inc.","2020, 2025, 2023, 2024"


In [15]:
display(cluster_medians)

log_assets  liabilities_to_assets  debt_to_assets  \
financial_flag cluster                                                      
Non-financial  0            18.060                  1.086           0.396   
               1            21.193                  0.454           0.236   
               2            18.684                  0.299           0.127   
               3            19.568                  0.652           0.244   
               4            22.807                  0.671           0.295   

                        debt_to_equity  equity_to_assets  cash_to_assets  \
financial_flag cluster                                                     
Non-financial  0                -0.805            -0.209           0.104   
               1                 0.483             0.485           0.117   
               2                 0.260             0.600           0.277   
               3                 0.836             0.296           0.049   
               4                 0.859             0.301           0.031   

                        net_income_to_assets  cfo_to_assets  \
financial_flag cluster                                        
Non-financial  0                      -0.277         -0.152   
               1                       0.049          0.094   
               2                      -0.258         -0.198   
               3                      -0.069         -0.009   
               4                       0.033          0.081   

                        revenue_to_assets  current_ratio  quick_ratio  \
financial_flag cluster                                                  
Non-financial  0                    0.452          0.846        0.394   
               1                    0.610          2.367        1.276   
               2                    0.240          4.310        2.130   
               3                    0.457          1.065        0.402   
               4                    0.491          1.011        0.351   

                        interest_coverage  fcf_to_debt  operating_margin  \
financial_flag cluster                                                     
Non-financial  0                   -7.906       -0.521            -0.529   
               1                    7.004        0.280             0.108   
               2                  -27.663       -1.220            -0.819   
               3                   -2.721       -0.093            -0.118   
               4                    3.280        0.203             0.128   

                        gross_margin  ebitda_margin  debt_to_ebitda  \
financial_flag cluster                                                
Non-financial  0               0.331         -0.459           9.587   
               1               0.376          0.148           2.375   
               2               0.407         -0.697           9.780   
               3               0.295         -0.063           9.131   
               4               0.329          0.186           3.373   

                        net_debt_to_ebitda  ebitda_interest_coverage  \
financial_flag cluster                                                 
Non-financial  0                     6.870                    -6.715   
               1                     1.249                    10.205   
               2                     3.997                   -23.475   
               3                     7.806                    -1.448   
               4                     2.864                     5.419   

                        leverage_risk  liquidity_risk  earnings_risk  \
financial_flag cluster                                                 
Non-financial  0                0.694           0.710          1.000   
               1                0.072           0.067          0.010   
               2                0.039           0.000          1.000   
               3                0.289           0.667          1.000   
               4             

In [16]:
display(feature_extremes)

,0.001,0.010,0.050,0.500,0.950,0.990,0.999
log_assets,13.861,14.320,15.561,20.124,24.670,26.001,28.156
liabilities_to_assets,0.004,0.035,0.076,0.510,1.413,4.431,17.012
debt_to_assets,0.000,0.000,0.005,0.246,0.759,1.401,4.898
debt_to_equity,-176.958,-18.145,-2.082,0.459,4.195,19.983,192.857
equity_to_assets,-15.753,-3.319,-0.472,0.383,0.885,1.107,1.474
cash_to_assets,0.000,0.000,0.004,0.109,0.700,0.972,1.350
net_income_to_assets,-10.512,-3.151,-1.244,-0.005,0.135,0.248,0.921
cfo_to_assets,-3.803,-1.833,-0.793,0.029,0.187,0.289,0.533
revenue_to_assets,0.000,0.000,0.012,0.468,1.905,3.384,8.017
current_ratio,0.003,0.056,0.351,1.869,12.338,27.939,81.451


In [17]:
display(industry_cluster_mix.head(50))

,financial_flag,cluster,major_sector,row_count,cluster_total,pct_of_cluster
2,Non-financial,0,Manufacturing / Industrials,1639,3350,0.489254
4,Non-financial,0,Services,1053,3350,0.314328
6,Non-financial,0,Wholesale / Retail,240,3350,0.071642
5,Non-financial,0,Transportation / Utilities,196,3350,0.058507
3,Non-financial,0,Mining / Energy,141,3350,0.042090
1,Non-financial,0,Construction,42,3350,0.012537
0,Non-financial,0,Agriculture,39,3350,0.011642
9,Non-financial,1,Manufacturing / Industrials,3635,7086,0.512983
11,Non-financial,1,Services,1735,7086,0.244849
13,Non-financial,1,Wholesale / Retail,599,7086,0.084533


## 6. Governance check: compare alternative k values

The selected five-cluster model is judged against nearby k alternatives for sanity, not automatic selection.

In [18]:
k_tests = evaluate_segments_k_range(
    df=model_df,
    model=model,
    segment_col=DEFAULT_SEGMENT_COL,
    k_values=range(2, 9),
    cluster_only_segments=DEFAULT_TARGET_SEGMENTS,
    min_rows=DEFAULT_MIN_ROWS_PER_SEGMENT,
)

k_tests

Initialization complete
Iteration 0, inertia 26656.650697296944.
Iteration 1, inertia 12917.2116750869.
Converged at iteration 1: center shift 8.895200517051369e-07 within tolerance 1.1780856404446312e-05.
Initialization complete
Iteration 0, inertia 17423.122941457183.
Iteration 1, inertia 10441.161787805922.
Iteration 2, inertia 9939.58923632402.
Iteration 3, inertia 9888.39172874585.
Iteration 4, inertia 9879.213022749957.
Iteration 5, inertia 9878.356577744931.
Converged at iteration 5: center shift 4.122120866268191e-06 within tolerance 1.1780856404446312e-05.
Initialization complete
Iteration 0, inertia 16818.984366352215.
Iteration 1, inertia 9941.288436152325.
Iteration 2, inertia 9886.30659343161.
Iteration 3, inertia 9878.90154739006.
Iteration 4, inertia 9878.343018709513.
Converged at iteration 4: center shift 3.4462444268348697e-06 within tolerance 1.1780856404446312e-05.
Initialization complete
Iteration 0, inertia 20519.869759003534.
Iteration 1, inertia 11973.0149792673

,segment,k,rows,features,feature_list,min_non_null_features,silhouette,calinski_harabasz,davies_bouldin
0,Non-financial,2,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.402994,17222.195827,1.094389
1,Non-financial,3,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.444888,16922.999579,0.917960
2,Non-financial,4,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.390730,15061.245505,1.156223
3,Non-financial,5,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.360739,13416.410830,1.211267
4,Non-financial,6,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.367774,12567.218070,1.255628
5,Non-financial,7,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.379717,12274.846712,1.168970
6,Non-financial,8,24003,6,"[structural_distress_risk, earnings_risk, operating_cashflow_risk, liquidity_risk, leverage_risk, debt_service_risk]",4,0.384871,11830.291236,1.113906


## 7. Save outputs and model artifact

Artifact schema/persistence lives in `artifacts.py`.

In [19]:
# ---------------------------------------------------------------------
# Save review / reporting outputs
# ---------------------------------------------------------------------
print("\nSaving review outputs...")

cluster_profile_ranked.to_csv(
    CURRENT_OUTPUT_DIR / f"cluster_profile_v3_by_{DEFAULT_SEGMENT_COL}.csv",
    index=False,
)

cluster_profile_ranked_with_reps.to_csv(
    CURRENT_OUTPUT_DIR / f"cluster_profile_with_reps_v3_by_{DEFAULT_SEGMENT_COL}.csv",
    index=False,
)

cluster_medians.to_csv(
    CURRENT_OUTPUT_DIR / f"cluster_medians_v3_by_{DEFAULT_SEGMENT_COL}.csv"
)

feature_extremes.to_csv(
    CURRENT_OUTPUT_DIR / "feature_extremes_v3.csv"
)

industry_cluster_mix.to_csv(
    CURRENT_OUTPUT_DIR / f"industry_cluster_mix_v3_by_{DEFAULT_SEGMENT_COL}.csv",
    index=False,
)

metrics_df.to_csv(
    CURRENT_OUTPUT_DIR / f"cluster_metrics_v3_by_{DEFAULT_SEGMENT_COL}.csv",
    index=False,
)

k_tests.to_csv(
    CURRENT_OUTPUT_DIR / f"cluster_k_tests_v3_by_{DEFAULT_SEGMENT_COL}.csv",
    index=False,
)

print("\nSaved review outputs to:", CURRENT_OUTPUT_DIR)


Saving review outputs...

Saved review outputs to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\credit_clustering_outputs_v3


In [20]:
artifact = build_credit_clustering_artifact(
    segment_artifacts=segment_artifacts,
    cluster_profile_ranked=cluster_profile_ranked,
    primary_segment="Non-financial",
    segment_col=DEFAULT_SEGMENT_COL,
    metrics_df=metrics_df,
    cluster_profile=cluster_profile,
    cluster_medians=cluster_medians,
    feature_extremes=feature_extremes,
    industry_cluster_mix=industry_cluster_mix,
    winsor_caps=None,
    artifact_version="v3_scorecard",
    notes=(
        "V3 KMeans++ k=5 model trained on bounded directional credit-risk factors. "
        "Debt-service risk includes EBITDA-enhanced diagnostics where available, "
        "with legacy coverage/FCF fallback when EBITDA is unavailable. "
        "Absolute size is excluded from clustering and retained only as diagnostic flags. "
        "Labels are rating-style interpretations, not formal credit ratings."
    ),
    extra_metadata={
        "model_name": "nonfinancial_credit_scorecard_kmeans_k5",
        "training_rows": int(len(clustered_panel[clustered_panel[DEFAULT_SEGMENT_COL] == "Non-financial"])),
        "source_notebook": "02_credit_clustering.ipynb",
    },
)

saved_path = save_artifact(
    artifact,
    ARTIFACT_PATH,
    segment="Non-financial",
)

print("Saved model artifact to:", saved_path)
summarize_artifact(artifact, segment="Non-financial")

Saved model artifact to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\saved_models\nonfinancial_credit_scorecard_kmeans_k5_v3.joblib


{'artifact_version': 'v3_scorecard',
 'primary_segment': 'Non-financial',
 'segment': 'Non-financial',
 'feature_cols': ['structural_distress_risk',
  'earnings_risk',
  'operating_cashflow_risk',
  'liquidity_risk',
  'leverage_risk',
  'debt_service_risk'],
 'n_clusters': 5,
 'cluster_labels': {1: '1 - Strong relative credit profile',
  4: '2 - Good credit profile',
  2: '3 - Leveraged / elevated risk profile',
  3: '4 - Weak credit profile',
  0: '5 - Distressed / near-default proxy'},
 'risk_rank': {1: 1, 4: 2, 2: 3, 3: 4, 0: 5},
 'cluster_sizes': {0: 3350, 1: 7086, 2: 6593, 3: 2630, 4: 4344},
 'silhouette': 0.36073933573901334,
 'calinski_harabasz': 13416.41083033897,
 'davies_bouldin': 1.2112674529852931}

## 8. Handoff to Notebook 03

Notebook 03 should load the saved artifact and use `score_companies()` plus `diagnostics.py` utilities for manual/private-company scoring.